In [1]:
##Imports

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import asyncio
import re
import os

In [2]:
##Loads clean cases; fetches each case page for press release and PDF links
##links scoped to <main> to skip nav/menu noise

BASE = "https://www.ftc.gov"
DATA_DIR = "/Users/nic/Documents/MM2/data"
UA = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"

df = pd.read_csv(f"{DATA_DIR}/ftc_cases_clean.csv")
print(df.shape)

def absolutize(href):
    return href if href.startswith("http") else BASE + href

def parse_case_links(html):
    main = BeautifulSoup(html, "html.parser").select_one("main")
    if main is None:
        return [], []
    pr = sorted(set(absolutize(a["href"]) for a in main.select('a[href*="/news-events/news/press-releases/"]')))
    pdf = sorted(set(absolutize(a["href"]) for a in main.select("a[href]") if ".pdf" in a["href"].lower()))
    return pr, pdf

async def fetch_case_links(urls, concurrency=5):
    results = {}
    async with async_playwright() as p:
        request = await p.request.new_context(extra_http_headers={"User-Agent": UA})
        sem = asyncio.Semaphore(concurrency)

        async def fetch(url):
            async with sem:
                try:
                    resp = await request.get(url, timeout=30000)
                    if resp.status != 200:
                        raise Exception(f"status {resp.status}")
                    pr, pdf = parse_case_links(await resp.text())
                    results[url] = {"press_release_urls": pr, "pdf_urls": pdf}
                except Exception:
                    results[url] = {"press_release_urls": [], "pdf_urls": []}
                await asyncio.sleep(0.2)

        await asyncio.gather(*[fetch(u) for u in urls])
        await request.dispose()
    return results

links = await fetch_case_links(df["url"].tolist())
df["press_release_urls"] = df["url"].map(lambda u: links[u]["press_release_urls"])
df["pdf_urls"] = df["url"].map(lambda u: links[u]["pdf_urls"])
print(f"cases with press releases: {(df['press_release_urls'].str.len() > 0).sum()}")
print(f"cases with PDFs: {(df['pdf_urls'].str.len() > 0).sum()}")

(339, 11)


cases with press releases: 333
cases with PDFs: 332


In [3]:
##Fetches each unique press release page and pulls the full body text

pr_urls = sorted(set(u for lst in df["press_release_urls"] for u in lst))
print(f"unique press releases: {len(pr_urls)}")

async def fetch_pr_text(urls, concurrency=5):
    results = {}
    async with async_playwright() as p:
        request = await p.request.new_context(extra_http_headers={"User-Agent": UA})
        sem = asyncio.Semaphore(concurrency)

        async def fetch(url):
            async with sem:
                try:
                    resp = await request.get(url, timeout=30000)
                    if resp.status != 200:
                        raise Exception(f"status {resp.status}")
                    body = BeautifulSoup(await resp.text(), "html.parser").select_one("main .field--name-body")
                    results[url] = body.get_text(" ", strip=True) if body else None
                except Exception:
                    results[url] = None
                await asyncio.sleep(0.2)

        await asyncio.gather(*[fetch(u) for u in urls])
        await request.dispose()
    return results

pr_texts = await fetch_pr_text(pr_urls)
df["press_release_text"] = df["press_release_urls"].map(
    lambda lst: "\n\n".join(t for u in lst if (t := pr_texts.get(u))) or None
)
print(f"cases with press release text: {df['press_release_text'].notna().sum()}")

unique press releases: 452


cases with press release text: 333


In [4]:
##Extracts penalty amounts from press release text; falls back to listing summary
##an amount only counts if a penalty keyword appears within 100 chars of it, so business-scale
##figures (revenue, loan portfolios, transaction volume) are ignored
##also skips statutory per-violation maximums ("Each violation ... may result in a civil penalty of up to $X")

BOILER_BEFORE = re.compile(r"(each violation|may result in|actions led to|returned)[^$]{0,60}$", re.I)
BOILER_AFTER = re.compile(r"^\s*(?:,\s*)?(?:per|for each)\s+(?:violation|day)", re.I)
PENALTY_KEYWORDS = re.compile(r"\bpay|\bpaid|penalt|\bfine[sd]?\b|redress|refund|judgment|disgorge|forfeit", re.I)

def extract_penalty(text):
    if not isinstance(text, str):
        return None
    amounts = []
    for m in re.finditer(r"\$(\d[\d,]*(?:\.\d+)?)\s*(billion|million|thousand)?", text, re.I):
        window = text[max(0, m.start() - 100):m.end() + 100]
        if not PENALTY_KEYWORDS.search(window):
            continue
        if BOILER_BEFORE.search(text[max(0, m.start() - 80):m.start()]) or BOILER_AFTER.match(text[m.end():m.end() + 30]):
            continue
        val = float(m.group(1).replace(",", ""))
        mult = {"billion": 1e9, "million": 1e6, "thousand": 1e3}.get((m.group(2) or "").lower(), 1)
        if val * mult >= 1000:  # drop app prices / fee mentions, no real penalty is under $1k
            amounts.append(val * mult)
    return max(amounts) if amounts else None

df["penalty_usd_enriched"] = df["press_release_text"].map(extract_penalty)
df["penalty_usd_enriched"] = df["penalty_usd_enriched"].fillna(df["penalty_usd"])
print(df[["penalty_usd", "penalty_usd_enriched", "press_release_text"]].notna().sum())

penalty_usd              25
penalty_usd_enriched    147
press_release_text      333
dtype: int64


In [5]:
##Saves enriched cases and per-PDF link table to .csv

df_out = df.copy()
df_out["press_release_urls"] = df_out["press_release_urls"].map("; ".join)
df_out["pdf_urls"] = df_out["pdf_urls"].map("; ".join)
df_out.to_csv(f"{DATA_DIR}/ftc_enriched.csv", index=False)

pdf_links = (
    df[["case_name", "pdf_urls"]]
    .explode("pdf_urls")
    .dropna(subset=["pdf_urls"])
    .rename(columns={"pdf_urls": "pdf_url"})
    .reset_index(drop=True)
)
pdf_links.to_csv(f"{DATA_DIR}/ftc_pdf_links.csv", index=False)

print("enriched:", df_out.shape)
print("pdf links:", pdf_links.shape)
df_out.head()

enriched: (339, 15)
pdf links: (2522, 2)


,case_name,url,date,case_type,matter_number,case_status,summary,tags,long_title,statutes,penalty_usd,press_release_urls,pdf_urls,press_release_text,penalty_usd_enriched
0,"Amazon.com, Inc., U.S. v.",https://www.ftc.gov/legal-library/browse/cases...,2026-06-30,Federal,2523024,Pending,Amazon will pay $2.25 million in civil penalti...,Consumer Protection; Bureau of Consumer Protec...,"UNITED STATES OF AMERICA, Plaintiff v. AMAZON....",FCRA,2250000.0,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/A...,Amazon will pay $2.25 million in civil penalti...,2250000.0
1,"FTC v Kochava, Inc.",https://www.ftc.gov/legal-library/browse/cases...,2026-06-26,Federal,NaN,Pending,The FTC will prohibit data broker Kochava and ...,Consumer Protection; Bureau of Consumer Protec...,"Federal Trade Commission, Plaintiff, V. Kochav...",NaN,NaN,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/1...,The Federal Trade Commission filed a lawsuit a...,NaN
2,"Illuminate Education, Inc., In the Matter of",https://www.ftc.gov/legal-library/browse/cases...,2026-06-05,Administrative,2223105,Under Order,The Federal Trade Commission will require educ...,Consumer Protection; Bureau of Consumer Protec...,"In the Matter of ILLUMINATE EDUCATION, INC., a...",NaN,NaN,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/2...,The Federal Trade Commission will require educ...,NaN
3,"Twitter, Inc., a corporation",https://www.ftc.gov/legal-library/browse/cases...,2026-06-03,Administrative,0923093,NaN,NaN,Consumer Protection; Bureau of Consumer Protec...,"In the Matter of Twitter, Inc.,a corporation",NaN,NaN,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/sites/default/files/docume...,Social networking service Twitter has agreed t...,NaN
4,"CMG Media Corporation, In the Matter of",https://www.ftc.gov/legal-library/browse/cases...,2026-05-21,Administrative,2423029,Pending,"The FTC will require Cox Media Group, MindSift...",Consumer Protection; Bureau of Consumer Protec...,In the Matter of CMG Media Corporation,NaN,930000.0,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/C...,The Federal Trade Commission will require Cox ...,930000.0
